# 02. Data Cleaning & Preprocessing
**Project:** Logistics & Delivery Delays Analysis (Olist)

## Objective
Transform the raw merged dataset into a high-quality, standardized format ready for analysis. This notebook documents **every cleaning step** with professional reasoning focused on the logistics problem statement.

---

## 1. Setup and Loading Data

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

PATH_MERGED = "../data/merged/olist_merged_data.csv"
df = pd.read_csv(PATH_MERGED)

print(f"Initial shape: {df.shape}")

Initial shape: (114092, 39)


## 2. Standardizing Datetime Columns

### **Why?**
Timestamps are loaded as strings by default. To perform any temporal analysis (e.g., calculating delivery duration, detecting delays), these must be converted into proper `datetime` objects. 

**Logistics Relevance:** We cannot calculate lead times without standardized date formats.

In [2]:
date_columns = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp'
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col])

print("Success: All temporal columns converted to datetime.")

Success: All temporal columns converted to datetime.


## 3. Handling Missing Values

### **3.1. Filtering by Order Status**
**Reasoning:** For an analysis dedicated to **Delivery Delays**, we must focus on orders that were actually delivered. Orders with status 'canceled', 'unavailable', or 'shipped' (but not yet delivered) lack the `order_delivered_customer_date` which is necessary to calculate actual delay.

**Action:** Retain only `delivered` orders.

In [3]:
initial_count = len(df)
df = df[df['order_status'] == 'delivered']
print(f"Rows removed (non-delivered): {initial_count - len(df)}")

Rows removed (non-delivered): 3252


### **3.2. Imputing Physical Dimensions**
**Reasoning:** Some products are missing weight and dimensions. Since these are physical items that were shipped, they must have weight and size. These factors influence shipping cost and speed.

**Action:** Replace missing values in physical attributes with the median value for that specific category, or the global median if the category is also missing.

In [4]:
physical_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

for col in physical_cols:
    df[col] = df[col].fillna(df[col].median())

print("Success: Missing physical dimensions imputed with median values.")

Success: Missing physical dimensions imputed with median values.


### **3.3. Dropping Irrelevant Null-Heavy Columns**
**Reasoning:** Columns like `review_comment_title` and `review_comment_message` have more than 80% missing values. While text mining is possible, for a structured logistics delay analysis, the `review_score` (which is present) is a sufficient proxy for customer satisfaction.

**Action:** Drop text-heavy columns to save memory and focus on quantitative metrics.

In [5]:
df.drop(columns=['review_comment_title', 'review_comment_message'], inplace=True, errors='ignore')
print("Dropped textual review columns.")

Dropped textual review columns.


## 4. Feature Engineering (Logistics Context)

### **Why?**
Raw timestamps are hard to interpret. We need to create **Lead Time** metrics to measure performance.

1. **Delivery Time**: Time from purchase to delivery (Total logistics time).
2. **Estimated Time**: Time promised to the customer.
3. **Delivery Delay**: Difference between actual and estimated (Positive = Late, Negative = Early).

In [6]:
# Total Delivery Time (Days)
df['actual_delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

# Estimated Delivery Time (Days)
df['estimated_delivery_days'] = (df['order_estimated_delivery_date'] - df['order_purchase_timestamp']).dt.days

# Delay (Days)
df['delivery_delay'] = df['actual_delivery_days'] - df['estimated_delivery_days']

# Binary Late Flag
df['is_late'] = df['delivery_delay'] > 0

print("Success: Logistics features engineered.")

Success: Logistics features engineered.


## 5. Handling Duplicates

### **Why?**
Because an order can have multiple items, the row count increased during extraction. For logistics delay analysis, we need per-order performance. However, different items in the same order might have different weights/categories.

**Action:** We will keep the granular data but ensure there are no redundant duplicate records caused by review duplication (sometimes customers leave multiple reviews for the same order).

In [7]:
# If any order has duplicate reviews, take the average score to avoid multiplying rows
cols_to_check = [c for c in df.columns if c not in ['review_score', 'review_id', 'review_answer_timestamp', 'review_creation_date']]
df = df.sort_values('review_creation_date').drop_duplicates(subset=['order_id', 'order_item_id'], keep='last')

print(f"Final deduplicated shape: {df.shape}")

Final deduplicated shape: (110197, 41)


## 6. Outlier Removal

### **Why?**
Extreme values (e.g., delivery taking 200 days) are often business process errors (e.g., system glitches or lost packages that were manually closed months later). Including them skews the average performance metrics.

**Action:** Remove the top 0.5% of extreme delivery delays.

In [8]:
q_upper = df['actual_delivery_days'].quantile(0.995)
df = df[df['actual_delivery_days'] <= q_upper]
print(f"Outliers removed above {q_upper} days.")

Outliers removed above 53.0 days.


## 7. Storage
The standardized dataset is now ready for Exploratory Data Analysis (EDA) and Statistical Modeling.

In [9]:
OUTPUT_PATH = "../data/processed/olist_cleaned_data.csv"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)
print(f"Standardised dataset successfully saved to: {OUTPUT_PATH}")

Standardised dataset successfully saved to: ../data/processed/olist_cleaned_data.csv
